##Building Results Fact - fact_session_results

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
 %run ../00-common/1.environment_config

In [0]:
%run ../00-common/4.gold_helpers

In [0]:
results_table = f"{catalog_name}.{silver_schema}.results"
sprints_table = f"{catalog_name}.{silver_schema}.sprints"
# target_table
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

## Reading results table and sprints table

In [0]:
from pyspark.sql import functions as F


results_df = (
    spark.table(results_table)
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("RACE"))
    .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)


sprints_df = (
    spark.table(sprints_table)
    .filter(F.col("batch_id") == v_batch_id)
    .withColumn("session_type", F.lit("SPRINT"))
    .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

## Union results and sprints

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
display(results_sprints_df)

#### Step 3 - Adding dervied columns
 1. `is_win` -> Indicates that the driver won the race
 1. `is_podium` -> Indicates that the driver scored a podium result (1, 2, 3)
 1. `has_points` -> Indicates that the driver has scored points

In [0]:
fact_session_results_df =(
    results_sprints_df
    .withColumns(
        {
            "is_win": F.col("final_position") == 1,
            "is_podium": F.col("final_position").between(1,3),
            "has_points": F.col("points") > 0
        }
    )
)

In [0]:
display(fact_session_results_df)

## Writing into the Gold Delta Table

In [0]:
results_fact_columns_to_update = [
    "grid_position",
    "completed_laps",
    "car_number",
    "points",
    "final_position",
    "final_position_text",
    "status",
    "is_win",
    "is_podium",
    "has_points"
]

write_to_gold(
    source_df=fact_session_results_df,
    target_table=target_table,
    merge_condition= """
    t.season = s.season 
    AND t.round = s.round 
    AND t.constructor_id = s.constructor_id 
    AND t.driver_id = s.driver_id
    AND t.session_type = s.session_type
    """,
    columns_to_update = results_fact_columns_to_update
)

In [0]:
spark.table(target_table).display()